# nuScenes Data Hierarchy Tree

This notebook shows the **hierarchical structure** of the nuScenes dataset:
- How scenes connect to samples
- How samples contain annotations
- How instances track objects across frames

We visualize the actual data flow with real examples.

## Step 1: Load NuScenes Dataset

In [ ]:
from nuscenes.nuscenes import NuScenes
import json

print("Loading nuScenes dataset...")
nutsc = NuScenes(version='v1.0-trainval', dataroot='/media/robotswithai/data1/ece1508/nuscenes_data', verbose=False)
print(f"✓ Loaded! Found {len(nusc.scene)} scenes")

## Step 2: Show Overall Statistics

In [ ]:
print("="*60)
print("NUSCENES DATABASE OVERVIEW")
print("="*60)

print(f"\nTotal scenes: {len(nusc.scene)}")
print(f"Total samples: {len(nusc.sample)}")
print(f"Total sample_data: {len(nusc.sample_data)}")
print(f"Total annotations: {len(nusc.sample_annotation)}")
print(f"Total instances: {len(nusc.instance)}")
print(f"Total categories: {len(nusc.category)}")
print(f"Total sensors: {len(nusc.sensor)}")

print("\n" + "="*60)

## Step 3: Pick One Scene and Show Its Structure

In [ ]:
# Pick scene #0
scene_idx = 0
my_scene = nusc.scene[scene_idx]

print(f"\n{'='*60}")
print(f"SCENE #{scene_idx} DETAILS")
print(f"{'='*60}")
print(f"\nScene name: {my_scene['name']}")
print(f"Description: {my_scene['description']}")
print(f"Number of samples: {my_scene['nbr_samples']}")
print(f"Duration: ~{my_scene['nbr_samples'] * 0.5} seconds")
print(f"\nFirst sample token: {my_scene['first_sample_token']}")
print(f"Last sample token: {my_scene['last_sample_token']}")
print(f"\n{'='*60}")

## Step 4: Visualize Scene → Sample → Annotation Tree

In [ ]:
# Get all samples in this scene
samples_in_scene = []
current_sample_token = my_scene['first_sample_token']
sample_count = 0

while current_sample_token:
    sample = nusc.get('sample', current_sample_token)
    samples_in_scene.append(sample)
    current_sample_token = sample['next']
    sample_count += 1

print(f"\nScene {scene_idx} → {sample_count} Samples\n")

# Show first 5 samples
for i, sample in enumerate(samples_in_scene[:5]):
    print(f"\n{'─'*60}")
    print(f"Sample #{i} (Timestamp: {sample['timestamp']})")
    print(f"Sample token: {sample['token']}")
    print(f"Number of annotations: {len(sample['anns'])}")
    print(f"\nAnnotations in this sample:")
    
    for j, ann_token in enumerate(sample['anns'][:3]):  # Show first 3 annotations
        annotation = nusc.get('sample_annotation', ann_token)
        instance = nusc.get('instance', annotation['instance_token'])
        category = nusc.get('category', annotation['category_token'])
        
        print(f"  [{j+1}] {category['name']:20} at ({annotation['translation'][0]:7.2f}, {annotation['translation'][1]:7.2f}, {annotation['translation'][2]:7.2f})")
        print(f"      Instance: {instance['name']} (appears in {instance['nbr_annotations']} frames)")
    
    if len(sample['anns']) > 3:
        print(f"  ... and {len(sample['anns']) - 3} more objects")

print(f"\n{'─'*60}")
print(f"... and {sample_count - 5} more samples")

## Step 5: Follow One Instance Through All Samples

In [ ]:
# Pick the first instance we found
first_annotation_in_scene = nusc.get('sample_annotation', samples_in_scene[0]['anns'][0])
instance_token = first_annotation_in_scene['instance_token']
my_instance = nusc.get('instance', instance_token)

print(f"\n{'='*60}")
print(f"TRACKING ONE INSTANCE ACROSS TIME")
print(f"{'='*60}")

print(f"\nInstance name: {my_instance['name']}")
print(f"Category: {nusc.get('category', my_instance['category_token'])['name']}")
print(f"Appears in {my_instance['nbr_annotations']} frames")
print(f"\nFirst appearance: {my_instance['first_annotation_token']}")
print(f"Last appearance: {my_instance['last_annotation_token']}")

print(f"\n{'─'*60}")
print(f"Position trajectory:\n")

# Traverse all annotations for this instance
current_ann_token = my_instance['first_annotation_token']
frame_count = 0

while current_ann_token:
    ann = nusc.get('sample_annotation', current_ann_token)
    sample = nusc.get('sample', ann['sample_token'])
    
    x, y, z = ann['translation']
    print(f"Frame {frame_count:2d}: Position=({x:7.2f}, {y:7.2f}, {z:6.2f}), Sample timestamp={sample['timestamp']}")
    
    current_ann_token = ann['next']
    frame_count += 1
    
    if frame_count > 10:  # Show first 10 frames
        print(f"... and {my_instance['nbr_annotations'] - 10} more frames")
        break

print(f"\n{'='*60}")

## Step 6: Show Sensor Data Links

In [ ]:
# Pick one sample and show its sensor data
sample = samples_in_scene[0]

print(f"\n{'='*60}")
print(f"SAMPLE #{0} - SENSOR DATA")
print(f"{'='*60}")

print(f"\nSample contains data from sensors:")
for sensor_name, sample_data_token in sample['data'].items():
    sample_data = nusc.get('sample_data', sample_data_token)
    print(f"  - {sensor_name:15} → file: {sample_data['filename']}")

print(f"\n{'='*60}")

## Step 7: Visualize the Hierarchy Diagram

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# Helper function to draw boxes
def draw_box(ax, x, y, width, height, text, color='lightblue'):
    box = FancyBboxPatch((x - width/2, y - height/2), width, height,
                          boxstyle="round,pad=0.1", 
                          edgecolor='black', facecolor=color, linewidth=2)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', fontsize=10, fontweight='bold', wrap=True)

def draw_arrow(ax, x1, y1, x2, y2, label=''):
    arrow = FancyArrowPatch((x1, y1), (x2, y2),
                            arrowstyle='->', mutation_scale=20, linewidth=2, color='black')
    ax.add_patch(arrow)
    if label:
        mid_x, mid_y = (x1 + x2) / 2, (y1 + y2) / 2
        ax.text(mid_x + 0.2, mid_y, label, fontsize=9, style='italic')

# Draw hierarchy
draw_box(ax, 5, 9, 2, 0.8, 'Database\n(850 scenes)', 'lightcoral')
draw_arrow(ax, 5, 8.6, 5, 8)

draw_box(ax, 5, 7.5, 2, 0.8, f'Scene #{scene_idx}\n({my_scene["nbr_samples"]} samples)', 'lightyellow')
draw_arrow(ax, 5, 7.1, 5, 6.3)

# Show 3 samples
for i in range(3):
    x_pos = 2 + i * 2.5
    draw_box(ax, x_pos, 5.8, 1.8, 0.8, f'Sample #{i}\n({len(samples_in_scene[i]["anns"])} annot.)', 'lightgreen')
    draw_arrow(ax, x_pos, 5.4, x_pos, 4.6)
    
    # Show annotations under each sample
    num_annot = min(3, len(samples_in_scene[i]['anns']))
    for j in range(num_annot):
        y_pos = 4 - j * 0.8
        ann_token = samples_in_scene[i]['anns'][j]
        ann = nusc.get('sample_annotation', ann_token)
        cat = nusc.get('category', ann['category_token'])
        draw_box(ax, x_pos, y_pos, 1.6, 0.6, f'{cat["name"][:15]}', 'lightblue')

# Add legend
ax.text(0.5, 1.5, 'Hierarchy Flow:', fontsize=12, fontweight='bold')
ax.text(0.5, 1.0, '• Database contains 850 Scenes', fontsize=10)
ax.text(0.5, 0.6, '• Each Scene has ~40 Samples (20 sec at 2Hz)', fontsize=10)
ax.text(0.5, 0.2, '• Each Sample has Annotations (labeled objects)', fontsize=10)

plt.title('nuScenes Hierarchical Structure', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("✓ Hierarchy diagram complete!")

## Step 8: Show Data Flow for Preprocessing

In [ ]:
print(f"\n{'='*60}")
print(f"HOW PREPROCESS.PY USES THIS STRUCTURE")
print(f"{'='*60}")

print("""
1. Loop through all 850 SCENES
   ↓
2. For each scene, get first_sample_token
   ↓
3. Chain through all SAMPLES using sample['next']
   ↓
4. For each sample, get all sample['anns'] (annotations)
   ↓
5. For each annotation, extract:
   - annotation['translation'] (x, y, z position)
   - annotation['instance_token'] (which object)
   ↓
6. Group by instance_token to track one object over time
   ↓
7. Collect 4 frames of history (2 seconds)
   + 12 frames of future (6 seconds)
   ↓
8. Save as tensors: train_x.pt (history) and train_y.pt (future)
""")

print(f"{'='*60}")

# Show concrete example
print("\nCONCRETE EXAMPLE:")
print(f"\nLook at Scene #{scene_idx}:")
print(f"  - Has {len(samples_in_scene)} samples")
print(f"  - Sample 0 has {len(samples_in_scene[0]['anns'])} objects")

if len(samples_in_scene) >= 4:
    print(f"\nTo extract a trajectory:")
    print(f"  1. Start at Sample 0 (frame -4)")
    print(f"  2. Get Sample 1 (frame -3)")
    print(f"  3. Get Sample 2 (frame -2)")
    print(f"  4. Get Sample 3 (frame -1) ← END OF HISTORY")
    print(f"  5. Get Sample 4 (frame +1)")
    print(f"  ...")
    print(f"  16. Get Sample 15 (frame +12) ← END OF FUTURE")
    print(f"\n  Extract all 16 positions for ONE instance_token")
    print(f"  → Save first 4 to train_x")
    print(f"  → Save next 12 to train_y")
    
    print(f"\nRepeat this for:")
    print(f"  - Every possible starting position")
    print(f"  - Every object in every sample")
    print(f"  - Every scene")
    print(f"  = 231,909 complete trajectories!")

## Summary: The Data Structure

```
nuScenes Database
├── Scene 0: "Construction, maneuver between trucks"
│   ├── Sample 0 (t=0.0s)
│   │   ├── Annotation: Car at (100.5, 45.2, 0.1)
│   │   ├── Annotation: Truck at (105.3, 40.1, 0.2)
│   │   └── Annotation: Motorcycle at (95.2, 50.5, -0.1)
│   ├── Sample 1 (t=0.5s)
│   │   ├── Annotation: Car at (101.2, 45.8, 0.1)
│   │   ├── Annotation: Truck at (106.1, 40.5, 0.2)
│   │   └── Annotation: Motorcycle at (95.9, 51.2, -0.1)
│   ├── Sample 2 (t=1.0s)
│   │   ├── Annotation: Car at (102.0, 46.5, 0.1)
│   │   ├── Annotation: Truck at (106.9, 40.9, 0.2)
│   │   └── Annotation: Motorcycle at (96.6, 51.9, -0.1)
│   └── ... 37 more samples
├── Scene 1: "Intersection, peds, waiting vehicle..."
│   └── ... 40 samples
└── ... 848 more scenes
```

**Key insight:** Each object (Car, Truck, etc.) appears as an **Annotation in multiple Samples**. By chaining these together, we get a **trajectory** (path through space over time).